# 08.10 · Solutions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/08-deep-learning/10-sequences-rnn-lstm/solutions.ipynb)

In [1]:
import os, warnings
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"        # hide TensorFlow's C++ start-up messages
warnings.filterwarnings("ignore")                # hide library deprecation notices
import numpy as np
import matplotlib.pyplot as plt
import keras
from keras import layers
keras.utils.set_random_seed(42)                  # same start every run (results still vary slightly by machine)
print("Keras", keras.__version__, "· backend", keras.backend.backend())

def rnn(xs, w_x=0.5, w_h=0.8, b=0.0):
    """A one-number RNN: h_t = tanh(w_x * x_t + w_h * h_(t-1) + b), starting from h_0 = 0."""
    h, states = 0.0, []
    for x in xs:
        h = np.tanh(w_x * x + w_h * h + b)
        states.append(round(float(h), 4))
    return states

def make_order_task(n, length=20, seed=0):
    """Sequences of random tokens 3-9 with one '1' and one '2'. Label 1 if '1' comes first."""
    rng = np.random.default_rng(seed)
    X = rng.integers(3, 10, (n, length))
    y = np.zeros(n, dtype=int)
    for i in range(n):
        p, q = rng.choice(length, 2, replace=False)
        X[i, p], X[i, q] = 1, 2
        y[i] = int(p < q)
    return X, y

def bag_model(length, vocab=10):
    """Averages the token embeddings: it cannot see order."""
    return keras.Sequential([keras.Input((length,)), layers.Embedding(vocab, 8),
                             layers.GlobalAveragePooling1D(),
                             layers.Dense(16, activation="relu"), layers.Dense(1, activation="sigmoid")])

def seq_model(length, cell=layers.LSTM, vocab=10):
    """Reads the tokens in order."""
    return keras.Sequential([keras.Input((length,)), layers.Embedding(vocab, 8),
                             cell(16), layers.Dense(1, activation="sigmoid")])

def fit_score(model, X, y, Xt, yt, epochs=15):
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    model.fit(X, y, epochs=epochs, batch_size=64, verbose=0)
    return model.evaluate(Xt, yt, verbose=0)[1]

Keras 3.15.1 · backend tensorflow


## Exercise 1

With $w_h = 0.2$: $h_1 = \tanh(0.5) = 0.4621$, then each step multiplies by about 0.2 inside tanh: $h_2 = \tanh(0.2 \times 0.4621) = \tanh(0.0924) = 0.0922$, $h_3 = 0.0184$, $h_4 = 0.0037$, $h_5 = 0.0007$. Gone after three steps.

With $w_h = 0.95$: $h_2 = \tanh(0.95 \times 0.4621) = \tanh(0.439) = 0.4128$, then 0.3732, 0.3404, 0.3126. Most of it survives five steps.

In [2]:
for w_h in [0.2, 0.95]:
    print(f"w_h = {w_h}:", rnn([1, 0, 0, 0, 0], w_h=w_h))

w_h = 0.2: [0.4621, 0.0922, 0.0184, 0.0037, 0.0007]
w_h = 0.95: [0.4621, 0.4128, 0.3732, 0.3404, 0.3126]


A larger recurrent weight means longer memory, but during training the same factor also controls whether gradients vanish or explode. An RNN cannot tune it freely; that is the problem LSTMs solve.

## Exercise 2

In [3]:
X, y = make_order_task(4000, seed=0)
X_test, y_test = make_order_task(1000, seed=1)
for name, cell in [("LSTM", layers.LSTM), ("GRU", layers.GRU)]:
    keras.utils.set_random_seed(42)
    m = seq_model(20, cell=cell)
    print(f"{name:<5} {m.count_params():>5,} weights   test accuracy {fit_score(m, X, y, X_test, y_test):.3f}")

LSTM  1,697 weights   test accuracy 1.000


GRU   1,345 weights   test accuracy 1.000


The GRU solves the task too, with fewer weights. In practice GRU and LSTM perform similarly; GRU is a little faster.

## Exercise 3

In [4]:
X, y = make_order_task(4000, length=100, seed=0)
X_test, y_test = make_order_task(1000, length=100, seed=1)
for name, cell in [("SimpleRNN", layers.SimpleRNN), ("LSTM", layers.LSTM)]:
    keras.utils.set_random_seed(42)
    acc = fit_score(seq_model(100, cell=cell), X, y, X_test, y_test)
    print(f"{name:<10} length 100: test accuracy {acc:.3f}")

SimpleRNN  length 100: test accuracy 0.700


LSTM       length 100: test accuracy 0.949


Over 100 steps the plain RNN loses track of which special token came first; the LSTM's gated memory carries it much further. This is vanishing memory in action.

## Exercise 4

In [5]:
import time
VOCAB = 10_000
for maxlen in [50, 200]:
    (a, b), (c, d) = keras.datasets.imdb.load_data(num_words=VOCAB)
    a, c = keras.utils.pad_sequences(a[:10_000], maxlen=maxlen), keras.utils.pad_sequences(c[:5_000], maxlen=maxlen)
    keras.utils.set_random_seed(42)
    m = keras.Sequential([keras.Input((maxlen,)), layers.Embedding(VOCAB, 32), layers.LSTM(32),
                          layers.Dense(1, activation="sigmoid")])
    m.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    m.fit(a, b[:10_000], epochs=3, batch_size=64, verbose=0)
    print(f"last {maxlen:>3} words: test accuracy {m.evaluate(c, d[:5_000], verbose=0)[1]:.3f}")

last  50 words: test accuracy 0.780


last 200 words: test accuracy 0.838


With only the last 50 words the model sees less of each review and accuracy falls. Reviews often state the verdict anywhere, not just at the end. Longer inputs help, but for an RNN they also mean more steps to remember across, which is one reason Transformers (Module 09) took over: they look at every word at once.